In [ ]:
import pandas as pd
import panel as pn
from sqlalchemy import create_engine, text

pn.extension()

USUARIO = 'postgres'
SENHA = '1234'
HOST = 'localhost'
PORTA = '5432'
BANCO = 'sistema_mentoria_fbd'
string_conexao = f'postgresql+psycopg2://{USUARIO}:{SENHA}@{HOST}:{PORTA}/{BANCO}'
engine = create_engine(string_conexao)

with engine.connect() as conexao:
    query_select = text("""
        SELECT e.matricula, e.cpf, e.nome_completo, e.email, m.ira, m.semestre_atual, m.especialidade
        FROM MENTOR m
        JOIN ESTUDANTE e ON m.matricula = e.matricula
        ORDER BY e.matricula DESC
    """)
    resultado = conexao.execute(query_select)
    df_mentores = pd.DataFrame(resultado.fetchall(), columns=resultado.keys())

input_matricula = pn.widgets.TextInput(name='Matrícula', width=150)
input_cpf = pn.widgets.TextInput(name='CPF', width=150)
input_nome = pn.widgets.TextInput(name='Nome Completo', width=250)
input_email = pn.widgets.TextInput(name='E-mail', width=250)
input_ira = pn.widgets.FloatInput(name='IRA', value=0.0, width=100)
input_semestre = pn.widgets.IntInput(name='Semestre Atual', value=1, width=100)
input_especialidade = pn.widgets.TextInput(name='Especialidade', width=200)

input_matricula_acao = pn.widgets.TextInput(name='Matrícula (Ação)', value="", width=200)
botao_salvar = pn.widgets.Button(name='Cadastrar Mentor', button_type='primary', width=200)
botao_atualizar = pn.widgets.Button(name='Atualizar Mentor', button_type='warning', width=200)
botao_deletar = pn.widgets.Button(name='Excluir Mentor', button_type='danger', width=200)

tabela_display = pn.pane.DataFrame(df_mentores, width=1100, max_rows=15)
mensagem = pn.pane.Markdown("")

def atualizar_tabela(conexao):
    res = conexao.execute(query_select)
    tabela_display.object = pd.DataFrame(res.fetchall(), columns=res.keys())

def funcao_inserir(event):
    try:
        with engine.begin() as conexao:
            q_estudante = text("INSERT INTO ESTUDANTE (matricula, cpf, nome_completo, email) VALUES (:mat, :cpf, :nome, :email)")
            conexao.execute(q_estudante, {"mat": input_matricula.value, "cpf": input_cpf.value, "nome": input_nome.value, "email": input_email.value})
            
            q_mentor = text("INSERT INTO MENTOR (matricula, ira, semestre_atual, especialidade) VALUES (:mat, :ira, :sem, :esp)")
            conexao.execute(q_mentor, {"mat": input_matricula.value, "ira": input_ira.value, "sem": input_semestre.value, "esp": input_especialidade.value})
            
            atualizar_tabela(conexao)
            mensagem.object = "✅ Mentor cadastrado com sucesso."
    except Exception as e:
        mensagem.object = f"❌ Erro ao cadastrar: {str(e)}"

def funcao_atualizar(event):
    try:
        with engine.begin() as conexao:
            q_estudante = text("UPDATE ESTUDANTE SET nome_completo = :nome, email = :email WHERE matricula = :mat_acao")
            conexao.execute(q_estudante, {"nome": input_nome.value, "email": input_email.value, "mat_acao": input_matricula_acao.value})
            
            q_mentor = text("UPDATE MENTOR SET ira = :ira, semestre_atual = :sem, especialidade = :esp WHERE matricula = :mat_acao")
            conexao.execute(q_mentor, {"ira": input_ira.value, "sem": input_semestre.value, "esp": input_especialidade.value, "mat_acao": input_matricula_acao.value})
            
            atualizar_tabela(conexao)
            mensagem.object = f"⚠️ Mentor {input_matricula_acao.value} atualizado."
    except Exception as e:
        mensagem.object = f"❌ Erro ao atualizar: {str(e)}"

def funcao_deletar(event):
    try:
        with engine.begin() as conexao:
            query = text("DELETE FROM ESTUDANTE WHERE matricula = :mat_acao")
            conexao.execute(query, {"mat_acao": input_matricula_acao.value})
            atualizar_tabela(conexao)
            mensagem.object = f"🗑️ Mentor {input_matricula_acao.value} excluído."
    except Exception as e:
        mensagem.object = f"❌ Erro ao excluir: {str(e)}"

botao_salvar.on_click(funcao_inserir)
botao_atualizar.on_click(funcao_atualizar)
botao_deletar.on_click(funcao_deletar)

layout_completo = pn.Column(
    pn.pane.Markdown("## Sistema de Mentoria - CRUD de Mentores"),
    mensagem,
    pn.Row(input_matricula, input_cpf, input_nome, input_email),
    pn.Row(input_ira, input_semestre, input_especialidade, botao_salvar),
    pn.layout.Divider(),
    pn.Row(input_matricula_acao, botao_atualizar, botao_deletar),
    pn.layout.Divider(),
    tabela_display
)

layout_completo.servable()